# 🏈 Predictor NFL v2

Agente de predicción de juegos NFL. Predice:
- **Puntos por equipo**, **puntos totales** y **probabilidad de victoria**
- **Yardas de pase / tierra / recepción por jugador**

Datos: [nflverse](https://github.com/nflverse) vía `nflreadpy` (gratis, sin API key).

**Novedades v2:**
| Mejora | Fuente | Qué aporta |
|---|---|---|
| EPA por jugada | `load_pbp()` | mide calidad ofensiva/defensiva mejor que yardas |
| Líneas de Vegas | `load_schedules()` | spread y total como features del modelo |
| Rosters actuales | `load_rosters()` | jugadores en su equipo 2026 real |
| Depth charts | `load_depth_charts()` | titulares oficiales (QB1, RB1-2, WR1-3, TE1-2) |
| Lesiones | `load_injuries()` | excluye jugadores Out/Doubtful |
| Registro | `predicciones.csv` | guarda predicciones y mide tu acierto real |

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import polars as pl
import nflreadpy as nfl
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error
from scipy.stats import norm

SEASONS = list(range(2020, 2026))   # historial de entrenamiento
TEMPORADA_TEST = 2025               # backtest
TEMPORADA_ACTUAL = 2026             # temporada a predecir
ROLL = 5                            # ventana de forma: últimos 5 juegos
ARCHIVO_PRED = 'predicciones.csv'   # registro de predicciones

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 60)

## 1. Carga de datos

Primera ejecución descarga ~300 MB de play-by-play (después queda cacheado).
Datos frescos durante la temporada: `nfl.clear_cache()` y re-ejecutar.

In [2]:
sched = nfl.load_schedules(SEASONS).to_pandas()
stats = nfl.load_player_stats(SEASONS).to_pandas()
sched = sched[sched['home_score'].notna()].copy()
stats = stats[stats['season_type'].isin(['REG', 'POST'])].copy()

# play-by-play: solo columnas de EPA (polars filtra antes de convertir)
pbp = pl.concat([
    nfl.load_pbp([yr]).select(['season','week','posteam','defteam','epa','pass','rush'])
    for yr in SEASONS
]).to_pandas()

# temporada actual: calendario oficial (con líneas de Vegas), rosters y depth charts
calendario = nfl.load_schedules([TEMPORADA_ACTUAL]).to_pandas()
roster = nfl.load_rosters([TEMPORADA_ACTUAL]).to_pandas()

depth_pl = nfl.load_depth_charts([TEMPORADA_ACTUAL])
depth = depth_pl.filter(pl.col('dt') == pl.col('dt').max().over('team')).to_pandas()

# lesiones: se publican durante la temporada
try:
    lesiones = nfl.load_injuries([TEMPORADA_ACTUAL]).to_pandas()
except Exception:
    lesiones = pd.DataFrame(columns=['team','week','gsis_id','full_name','report_status'])
    print('⚠ Reportes de lesión 2026 aún no publicados (salen con la temporada)')

print(f'Juegos: {len(sched)} | Jugador-semanas: {len(stats)} | Jugadas pbp: {len(pbp)}')
print(f'Calendario {TEMPORADA_ACTUAL}: {len(calendario)} juegos | Roster: {len(roster)} | Depth chart: {len(depth)} (snapshot {depth.dt.max()[:10] if len(depth) else "-"})')

⚠ Reportes de lesión 2026 aún no publicados (salen con la temporada)
Juegos: 1693 | Jugador-semanas: 112447 | Jugadas pbp: 294989
Calendario 2026: 272 juegos | Roster: 2930 | Depth chart: 3172 (snapshot 2026-07-09)


## 2. Tabla equipo-juego y features

Cada juego → 2 filas (una por equipo). Features:
- **Rolling (últimos 5 juegos, sin fuga de datos):** puntos a favor/en contra, yardas pase/tierra a favor/en contra, **EPA ofensivo/defensivo** (total, pase, carrera)
- **Del juego:** localía, descanso, **spread y total de Vegas**

In [3]:
base = ['season','week','game_id','gameday']

home = sched[base + ['home_team','away_team','home_score','away_score','home_rest','spread_line','total_line']].copy()
home.columns = base + ['team','opp','pts','pts_perm','rest','vegas_spread','vegas_total']
home['es_local'] = 1

away = sched[base + ['away_team','home_team','away_score','home_score','away_rest','spread_line','total_line']].copy()
away.columns = base + ['team','opp','pts','pts_perm','rest','vegas_spread','vegas_total']
away['vegas_spread'] = -away['vegas_spread']   # spread desde la perspectiva del equipo
away['es_local'] = 0

tg = pd.concat([home, away], ignore_index=True)

# yardas por equipo-juego
yardas = stats.groupby(['season','week','team'], as_index=False).agg(
    pass_yds=('passing_yards','sum'), rush_yds=('rushing_yards','sum'))
tg = tg.merge(yardas, on=['season','week','team'], how='left')
tg = tg.merge(yardas.rename(columns={'team':'opp','pass_yds':'pass_yds_perm','rush_yds':'rush_yds_perm'}),
              on=['season','week','opp'], how='left')

# EPA por equipo-juego (ofensiva y defensiva, con desglose pase/carrera)
pj = pbp[pbp['epa'].notna() & pbp['posteam'].notna()].copy()
epa_of = pj.groupby(['season','week','posteam'], as_index=False).agg(epa_of=('epa','mean'))
epa_pase = pj[pj['pass'] == 1].groupby(['season','week','posteam'], as_index=False).agg(epa_pase=('epa','mean'))
epa_carr = pj[pj['rush'] == 1].groupby(['season','week','posteam'], as_index=False).agg(epa_carr=('epa','mean'))
epa_def = pj.groupby(['season','week','defteam'], as_index=False).agg(epa_def=('epa','mean'))

tg = (tg.merge(epa_of, left_on=['season','week','team'], right_on=['season','week','posteam'], how='left')
        .merge(epa_pase, left_on=['season','week','team'], right_on=['season','week','posteam'], how='left', suffixes=('','_x1'))
        .merge(epa_carr, left_on=['season','week','team'], right_on=['season','week','posteam'], how='left', suffixes=('','_x2'))
        .merge(epa_def, left_on=['season','week','team'], right_on=['season','week','defteam'], how='left')
        .drop(columns=['posteam','posteam_x1','posteam_x2','defteam']))

tg = tg.sort_values(['team','season','week']).reset_index(drop=True)

STAT_COLS = ['pts','pts_perm','pass_yds','rush_yds','pass_yds_perm','rush_yds_perm',
             'epa_of','epa_pase','epa_carr','epa_def']

def rolling_shift(s):
    # promedio de los ROLL juegos ANTERIORES (shift evita usar el juego actual)
    return s.shift(1).rolling(ROLL, min_periods=2).mean()

for c in STAT_COLS:
    tg[f'{c}_r'] = tg.groupby('team')[c].transform(rolling_shift)

FEAT_TEAM = [f'{c}_r' for c in STAT_COLS]
print(f'Tabla equipo-juego: {len(tg)} filas, {len(FEAT_TEAM)} features rolling + Vegas + contexto')

Tabla equipo-juego: 3386 filas, 10 features rolling + Vegas + contexto


## 3. Modelo de puntos por equipo

XGBoost: forma propia + forma del rival + localía + descanso + **líneas de Vegas**.

In [4]:
opp_feats = tg[['season','week','team'] + FEAT_TEAM].rename(
    columns={'team':'opp', **{f: f + '_vs' for f in FEAT_TEAM}})
m = tg.merge(opp_feats, on=['season','week','opp'], how='left')

FEATS = FEAT_TEAM + [f + '_vs' for f in FEAT_TEAM] + ['es_local','rest','vegas_spread','vegas_total']
m = m.dropna(subset=FEATS + ['pts'])

train = m[m['season'] < TEMPORADA_TEST]
test  = m[m['season'] == TEMPORADA_TEST]

modelo_pts = XGBRegressor(n_estimators=400, learning_rate=0.04, max_depth=4,
                          subsample=0.8, colsample_bytree=0.8, random_state=42)
modelo_pts.fit(train[FEATS], train['pts'])

test = test.copy()
test['pts_pred'] = modelo_pts.predict(test[FEATS])
print(f'MAE puntos por equipo ({TEMPORADA_TEST}): {mean_absolute_error(test.pts, test.pts_pred):.2f}')

imp = pd.Series(modelo_pts.feature_importances_, index=FEATS).sort_values(ascending=False)
print('\nTop 8 features:')
print(imp.head(8).round(3).to_string())

MAE puntos por equipo (2025): 7.47

Top 8 features:
vegas_spread     0.131
vegas_total      0.074
epa_of_r         0.053
es_local         0.047
epa_pase_r_vs    0.039
epa_carr_r_vs    0.037
epa_carr_r       0.037
epa_pase_r       0.037


In [5]:
# evaluación a nivel juego: total, ganador, y comparación contra Vegas
locales = test[test['es_local'] == 1][['game_id','team','opp','pts','pts_pred','vegas_spread','vegas_total']]
visitas = test[test['es_local'] == 0][['game_id','pts','pts_pred']].rename(
    columns={'pts':'pts_v','pts_pred':'pts_pred_v'})
juegos = locales.merge(visitas, on='game_id')

juegos['total_real'] = juegos['pts'] + juegos['pts_v']
juegos['total_pred'] = juegos['pts_pred'] + juegos['pts_pred_v']
juegos['dif_pred'] = juegos['pts_pred'] - juegos['pts_pred_v']
dif_real = juegos['pts'] - juegos['pts_v']
juegos['acierto'] = np.sign(juegos['dif_pred']) == np.sign(dif_real)
acierto_vegas = (np.sign(juegos['vegas_spread']) == np.sign(dif_real)).mean()

SIGMA_DIF = (juegos['dif_pred'] - dif_real).std()

print(f'MAE puntos totales: {mean_absolute_error(juegos.total_real, juegos.total_pred):.2f}')
print(f'Acierto de ganador — modelo: {juegos.acierto.mean():.1%}  |  Vegas: {acierto_vegas:.1%}')
print(f'Sigma diferencial: {SIGMA_DIF:.1f}')

MAE puntos totales: 10.89
Acierto de ganador — modelo: 59.6%  |  Vegas: 65.6%
Sigma diferencial: 12.5


## 4. Modelos de props por jugador

| Prop | Posiciones | Features |
|---|---|---|
| Yardas de pase | QB | forma (yardas, intentos) + defensa aérea rival |
| Yardas por tierra | RB, QB, FB | forma (yardas, acarreos) + defensa terrestre rival |
| Yardas recibidas | WR, TE, RB, FB | forma (yardas, targets, cuota) + defensa aérea rival |

In [6]:
ps = stats[['player_id','player_display_name','position','season','week','team','opponent_team',
            'passing_yards','attempts','rushing_yards','carries',
            'receiving_yards','targets','target_share']].copy()
ps = ps.rename(columns={'player_display_name':'jugador','opponent_team':'opp'})
ps = ps.sort_values(['player_id','season','week']).reset_index(drop=True)

PCOLS = ['passing_yards','attempts','rushing_yards','carries','receiving_yards','targets','target_share']
for c in PCOLS:
    ps[f'{c}_r'] = ps.groupby('player_id')[c].transform(rolling_shift)

defensa = tg[['season','week','team','pass_yds_perm_r','rush_yds_perm_r']].rename(
    columns={'team':'opp','pass_yds_perm_r':'def_pase','rush_yds_perm_r':'def_tierra'})
ps = ps.merge(defensa, on=['season','week','opp'], how='left')

PROPS = {
    'pase':      dict(target='passing_yards',   pos=['QB'],
                      feats=['passing_yards_r','attempts_r','def_pase']),
    'tierra':    dict(target='rushing_yards',   pos=['RB','QB','FB'],
                      feats=['rushing_yards_r','carries_r','def_tierra']),
    'recepcion': dict(target='receiving_yards', pos=['WR','TE','RB','FB'],
                      feats=['receiving_yards_r','targets_r','target_share_r','def_pase']),
}

modelos_prop, sigmas_prop = {}, {}
for nombre, cfg in PROPS.items():
    d = ps[ps['position'].isin(cfg['pos'])].dropna(subset=cfg['feats'] + [cfg['target']])
    tr, te = d[d['season'] < TEMPORADA_TEST], d[d['season'] == TEMPORADA_TEST]
    mod = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=4,
                       subsample=0.8, colsample_bytree=0.8, random_state=42)
    mod.fit(tr[cfg['feats']], tr[cfg['target']])
    pred = mod.predict(te[cfg['feats']])
    modelos_prop[nombre] = mod
    sigmas_prop[nombre] = float((te[cfg['target']] - pred).std())
    print(f'MAE yardas {nombre} ({TEMPORADA_TEST}): {mean_absolute_error(te[cfg.get("target")], pred):.1f}  ({len(te)} jugador-juegos)')

MAE yardas pase (2025): 72.5  (664 jugador-juegos)
MAE yardas tierra (2025): 17.2  (2309 jugador-juegos)


MAE yardas recepcion (2025): 16.3  (5481 jugador-juegos)


## 5. Función de predicción

Usa **roster y depth chart oficial 2026** (titulares: QB1, RB1-2, WR1-3, TE1-2), excluye lesionados
(Out/Doubtful) cuando hay reporte, y toma las **líneas de Vegas del calendario oficial**.

```python
predecir_juego('KC', 'BUF')          # busca el juego en el calendario 2026
predecir_juego('KC', 'BUF', week=5)  # si juegan 2 veces, especifica semana
```

In [7]:
LIM_DEPTH = {'QB': 1, 'RB': 2, 'WR': 3, 'TE': 2}
PROPS_POR_POS = {'QB': ['pase','tierra'], 'RB': ['tierra','recepcion'],
                 'WR': ['recepcion'], 'TE': ['recepcion']}

def _snapshot_equipo(team):
    d = tg[tg['team'] == team].sort_values(['season','week']).tail(ROLL)
    if d.empty:
        raise ValueError(f'Equipo desconocido: {team}')
    return {f'{c}_r': d[c].mean() for c in STAT_COLS}

def _fila_equipo(team, opp, es_local, rest, vegas_spread, vegas_total):
    snap, snap_vs = _snapshot_equipo(team), _snapshot_equipo(opp)
    fila = {**snap, **{k + '_vs': v for k, v in snap_vs.items()},
            'es_local': es_local, 'rest': rest,
            'vegas_spread': vegas_spread, 'vegas_total': vegas_total}
    return pd.DataFrame([fila])[FEATS]

def _snapshot_jugador(pid):
    d = ps[ps['player_id'] == pid].sort_values(['season','week']).tail(ROLL)
    if len(d) < 2:
        return None   # sin historial suficiente (ej. novato)
    return {f'{c}_r': d[c].mean() for c in PCOLS}

def _titulares(team, week=None):
    dc = depth[(depth['team'] == team) & (depth['pos_abb'].isin(LIM_DEPTH))].copy()
    dc = dc[dc['pos_rank'] <= dc['pos_abb'].map(LIM_DEPTH)]
    if week is not None and len(lesiones):
        fuera = set(lesiones[(lesiones['team'] == team) & (lesiones['week'] == week) &
                             (lesiones['report_status'].isin(['Out','Doubtful']))]['gsis_id'])
        descartados = dc[dc['gsis_id'].isin(fuera)]['player_name'].tolist()
        if descartados:
            print(f'  Lesionados fuera ({team}): {", ".join(descartados)}')
        dc = dc[~dc['gsis_id'].isin(fuera)]
    return dc[['gsis_id','player_name','pos_abb']].drop_duplicates('gsis_id')

def _props_equipo(team, opp_def, week=None):
    filas, sin_historial = [], []
    for _, j in _titulares(team, week).iterrows():
        snap = _snapshot_jugador(j['gsis_id'])
        if snap is None:
            sin_historial.append(f"{j['player_name']} ({j['pos_abb']})")
            continue
        for prop in PROPS_POR_POS[j['pos_abb']]:
            cfg = PROPS[prop]
            fila = {**snap, 'def_pase': opp_def['def_pase'], 'def_tierra': opp_def['def_tierra']}
            y = float(modelos_prop[prop].predict(pd.DataFrame([fila])[cfg['feats']])[0])
            filas.append(dict(equipo=team, jugador=j['player_name'], pos=j['pos_abb'],
                              prop=f'yardas_{prop}', prediccion=round(y, 1),
                              rango_68pct=f'{max(0, y - sigmas_prop[prop]):.0f}–{y + sigmas_prop[prop]:.0f}'))
    if sin_historial:
        print(f'  Sin historial NFL ({team}): {", ".join(sin_historial)}')
    return filas

def predecir_juego(local, visitante, week=None):
    g = calendario[(calendario['home_team'] == local) & (calendario['away_team'] == visitante)]
    if week is not None:
        g = g[g['week'] == week]
    if len(g):
        g = g.iloc[0]
        week = int(g['week'])
        spread, total = float(g['spread_line']), float(g['total_line'])
        rest_l, rest_v = float(g['home_rest']), float(g['away_rest'])
        print(f'Juego oficial: semana {week} ({g["gameday"]}) | Vegas: spread {spread:+.1f}, total {total}')
    else:
        spread, total, rest_l, rest_v = 0.0, 44.5, 7, 7
        print('⚠ Juego no está en el calendario oficial — usando líneas neutras')

    pts_l = float(modelo_pts.predict(_fila_equipo(local, visitante, 1, rest_l, spread, total))[0])
    pts_v = float(modelo_pts.predict(_fila_equipo(visitante, local, 0, rest_v, -spread, total))[0])
    p_local = float(norm.cdf((pts_l - pts_v) / SIGMA_DIF))

    print(f'=== {visitante} @ {local} ===')
    print(f'Marcador predicho:  {local} {pts_l:.1f} — {visitante} {pts_v:.1f}')
    print(f'Puntos totales:     {pts_l + pts_v:.1f}')
    print(f'Prob. victoria:     {local} {p_local:.0%}  |  {visitante} {1 - p_local:.0%}')

    def_l, def_v = _snapshot_equipo(local), _snapshot_equipo(visitante)
    props = (_props_equipo(local, {'def_pase': def_v['pass_yds_perm_r'], 'def_tierra': def_v['rush_yds_perm_r']}, week)
           + _props_equipo(visitante, {'def_pase': def_l['pass_yds_perm_r'], 'def_tierra': def_l['rush_yds_perm_r']}, week))
    df = pd.DataFrame(props).sort_values(['equipo','prop','prediccion'], ascending=[True, True, False])
    return df.reset_index(drop=True)

In [8]:
predecir_juego('KC', 'BUF')

⚠ Juego no está en el calendario oficial — usando líneas neutras
=== BUF @ KC ===
Marcador predicho:  KC 25.0 — BUF 24.0
Puntos totales:     49.1
Prob. victoria:     KC 53%  |  BUF 47%
  Sin historial NFL (KC): Emmett Johnson (RB)


,equipo,jugador,pos,prop,prediccion,rango_68pct
0,BUF,Josh Allen,QB,yardas_pase,228.1,137–319
1,BUF,Khalil Shakir,WR,yardas_recepcion,61.0,38–84
2,BUF,DJ Moore,WR,yardas_recepcion,39.0,16–62
3,BUF,Dalton Kincaid,TE,yardas_recepcion,30.5,7–54
4,BUF,Dawson Knox,TE,yardas_recepcion,23.6,0–47
5,BUF,Joshua Palmer,WR,yardas_recepcion,15.1,0–38
6,BUF,James Cook III,RB,yardas_recepcion,14.8,0–38
7,BUF,Ray Davis,RB,yardas_recepcion,13.5,0–37
8,BUF,James Cook III,RB,yardas_tierra,64.1,38–90
9,BUF,Josh Allen,QB,yardas_tierra,39.2,13–65


## 6. Predecir una semana completa (calendario oficial)

In [9]:
def predecir_semana(week):
    filas = []
    for _, g in calendario[calendario['week'] == week].iterrows():
        local, visita = g['home_team'], g['away_team']
        spread, total = float(g['spread_line']), float(g['total_line'])
        pts_l = float(modelo_pts.predict(_fila_equipo(local, visita, 1, g['home_rest'], spread, total))[0])
        pts_v = float(modelo_pts.predict(_fila_equipo(visita, local, 0, g['away_rest'], -spread, total))[0])
        p_local = float(norm.cdf((pts_l - pts_v) / SIGMA_DIF))
        filas.append(dict(
            season=TEMPORADA_ACTUAL, week=week, fecha=g['gameday'],
            local=local, visitante=visita,
            pred_local=round(pts_l, 1), pred_visita=round(pts_v, 1),
            total_pred=round(pts_l + pts_v, 1), total_vegas=total,
            # spread positivo = local favorito (misma convención que Vegas)
            spread_pred=round(pts_l - pts_v, 1), spread_vegas=spread,
            ganador=local if p_local >= 0.5 else visita,
            prob=round(max(p_local, 1 - p_local), 2)))
    return pd.DataFrame(filas)

predecir_semana(1)

,season,week,fecha,local,visitante,pred_local,pred_visita,total_pred,total_vegas,spread_pred,spread_vegas,ganador,prob
0,2026,1,2026-09-09,SEA,NE,28.1,18.4,46.5,44.5,9.6,3.5,SEA,0.78
1,2026,1,2026-09-10,LA,SF,25.3,24.0,49.3,48.5,1.3,3.0,LA,0.54
2,2026,1,2026-09-13,CAR,CHI,20.8,21.9,42.7,44.5,-1.1,-2.5,CHI,0.54
3,2026,1,2026-09-13,CIN,TB,27.3,24.8,52.1,50.5,2.5,3.5,CIN,0.58
4,2026,1,2026-09-13,DET,NO,27.0,25.1,52.1,48.5,2.0,7.0,DET,0.56
5,2026,1,2026-09-13,HOU,BUF,22.0,23.6,45.6,45.5,-1.7,-1.5,BUF,0.55
6,2026,1,2026-09-13,IND,BAL,20.6,29.8,50.4,49.5,-9.1,-3.5,BAL,0.77
7,2026,1,2026-09-13,JAX,CLE,27.9,16.9,44.7,40.5,11.0,7.5,JAX,0.81
8,2026,1,2026-09-13,PIT,ATL,22.5,19.8,42.3,42.5,2.7,3.0,PIT,0.59
9,2026,1,2026-09-13,TEN,NYJ,21.1,19.5,40.6,39.5,1.7,3.0,TEN,0.55


## 7. Registro de predicciones

- `guardar_predicciones(week)` — predice la semana y la anexa a `predicciones.csv`
- `evaluar_predicciones()` — compara lo guardado contra resultados reales y contra Vegas

Flujo semanal en temporada: `nfl.clear_cache()` → re-ejecutar notebook → `guardar_predicciones(week)` → después de los juegos, `evaluar_predicciones()`.

In [10]:
import os

def guardar_predicciones(week):
    df = predecir_semana(week)
    df['fecha_prediccion'] = pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')
    existe = os.path.exists(ARCHIVO_PRED)
    if existe:
        prev = pd.read_csv(ARCHIVO_PRED)
        # no duplicar: elimina predicciones previas de la misma semana
        prev = prev[~((prev['season'] == TEMPORADA_ACTUAL) & (prev['week'] == week))]
        df = pd.concat([prev, df], ignore_index=True)
    df.to_csv(ARCHIVO_PRED, index=False)
    print(f'{len(df[df.week == week])} juegos de semana {week} guardados en {ARCHIVO_PRED}')
    return df[df['week'] == week]

def evaluar_predicciones():
    if not os.path.exists(ARCHIVO_PRED):
        print(f'No existe {ARCHIVO_PRED} — usa guardar_predicciones(week) primero')
        return None
    log = pd.read_csv(ARCHIVO_PRED)
    res = nfl.load_schedules([TEMPORADA_ACTUAL]).to_pandas()
    res = res[res['home_score'].notna()][['week','home_team','away_team','home_score','away_score']]
    ev = log.merge(res, left_on=['week','local','visitante'],
                   right_on=['week','home_team','away_team'])
    if ev.empty:
        print('Aún no hay resultados para las predicciones guardadas')
        return None
    dif_real = ev['home_score'] - ev['away_score']
    ev['ganador_real'] = np.where(dif_real > 0, ev['local'], ev['visitante'])
    ev['acierto'] = ev['ganador'] == ev['ganador_real']
    acierto_vegas = (np.sign(ev['spread_vegas']) == np.sign(dif_real)).mean()
    print(f'Juegos evaluados: {len(ev)}')
    print(f'Acierto ganador — modelo: {ev.acierto.mean():.1%}  |  Vegas: {acierto_vegas:.1%}')
    print(f'MAE total: {(ev.total_pred - (ev.home_score + ev.away_score)).abs().mean():.2f}')
    print(f'MAE spread: {(ev.spread_pred - dif_real).abs().mean():.2f}')
    return ev[['week','local','visitante','pred_local','pred_visita','home_score','away_score','ganador','ganador_real','acierto','prob']]

# ejemplo (bórralo si no quieres guardar aún):
guardar_predicciones(1)
evaluar_predicciones()

16 juegos de semana 1 guardados en predicciones.csv
Aún no hay resultados para las predicciones guardadas


## 8. Notas para la temporada 2026

**Flujo semanal:**
1. `nfl.clear_cache()` y re-ejecuta todo el notebook (datos y lesiones frescos)
2. Cuando haya juegos 2026 jugados: `SEASONS = list(range(2020, 2027))` para que la forma reciente incluya 2026
3. `guardar_predicciones(week)` antes de los juegos; `evaluar_predicciones()` después

**Limitaciones honestas:**
- Hoy (offseason) la "forma" viene de fin de 2025; los rosters ya son 2026 pero los números individuales son del equipo anterior del jugador. Se corrige solo conforme corran semanas de 2026.
- Novatos sin historial NFL se omiten en props (se avisa).
- Con Vegas como feature, el modelo converge hacia Vegas — su valor está en detectar discrepancias grandes y en las props, no en "ganarle" a Vegas de forma sistemática.